# AHR v2 — play and watch on 9×9

Run All in the **AHR Go** kernel, choose Black or White, click **New game**, then click a board intersection. AHR replies automatically. **AI move** advances one move in watch mode. **Pass**, **Undo turn**, and **Resume saved game** are available. **Reload checkpoint** starts a new game with the selected latest/best checkpoint, so a running trainer never changes the opponent during your game.

Purpose: play against the fresh Scout/Worker network and batched search. Inputs: notebooks 00–04 tagged definitions and the active run's checkpoint. Outputs: interactive board and saved human games. Run All takes about 15–45 seconds on CPU and performs no training. Default AI search budget is 1.5 seconds; one atomic batch may finish slightly after that limit. Increase visits/time for more search. Search is bounded and does not prove the best move or certify playing strength.

Training checkpoints: `checkpoints/<training_run_id>/`. Each human game retains its immutable opponent snapshot and complete replay in `checkpoints/ahr-v2-human-games/<game_id>/`. **Exact resume:** Run All in a fresh kernel, then click **Resume saved game**, or call `Go2PlaySession.restore(PROJECT, saved_game_path)`. State, positional-superko history, replay, search settings and model bytes are verified before continuing. Source changes, corruption and stale concurrent writes are rejected. No service is published.

Rules: area scoring, 7.5 komi, no suicide, positional superko; two consecutive passes end the game. Dead stones must be captured before passing. There is no automatic dead-stone removal. Partial-board area is a current-board count, not the final outcome. Coordinates skip I: A–H, J.

## Load definitions

In [ ]:
from pathlib import Path
import nbformat
PROJECT=Path.cwd().resolve()
for candidate in (PROJECT,*PROJECT.parents):
    if (candidate/'experiments/ahr_v2/00_contracts_and_recovery.ipynb').is_file():PROJECT=candidate;break
else:raise RuntimeError('Open this notebook inside the AHR Go workspace.')
for cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'00:ahr-export','exec'),globals())
for name in ('01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb'):
    go2_import_exports(PROJECT/GO2_ROOT/name,globals())
training_notebook=PROJECT/GO2_ROOT/'04_selfplay_and_training.ipynb'
if training_notebook.exists():go2_import_exports(training_notebook,globals())
torch.set_num_threads(2)
PLAY_DEVICE='cpu'  # Leaves GPU 1 available to the detached trainer.
CHECKPOINT_CHOICE='latest'


## Frozen opponent and human-game recovery

In [ ]:
import html, ipywidgets as widgets
from IPython.display import display,HTML
GO2_PLAY_SCHEMA='ahr-go-v2-human-game-v1'

def go2_play_sources(project):
    result=go2_source_identity(project)
    for name in ('04_selfplay_and_training.ipynb','06_play_and_watch.ipynb'):
        path=Path(project)/GO2_ROOT/name
        if path.exists():result[str(path.relative_to(project))]=go2_code_sha(path)
    return result

def go2_play_coord(action,size=9):
    if action==size*size:return 'Pass'
    row,col=divmod(int(action),size)
    return 'ABCDEFGHJKLMNOPQRST'[col]+str(size-row)

def go2_play_active_run(project):
    for relative in ('config/ahr_v2_active_run.json','reports/ahr_v2/active_run.json','config/ahr_v2_training.json'):
        path=Path(project)/relative
        if path.exists():
            data=json.loads(path.read_text());run_id=data.get('run_id',data.get('config',{}).get('run_id'))
            if run_id:return str(run_id)
    return None

def go2_play_load_model(project,which='latest',device='cpu',allow_smoke=False):
    project=Path(project);run_id=go2_play_active_run(project)
    model=None
    if run_id is not None:
        if 'go2_load_player' not in globals():raise RuntimeError('The active run requires notebook 04. Run All again after it is present.')
        directory=project/'checkpoints'/run_id
        if (directory/'manifest.json').exists():
            model,metadata=go2_load_player(project=project,run_id=run_id,which=which,device=device)
            if metadata.get('is_smoke') and not allow_smoke:raise RuntimeError('Disposable smoke checkpoints are not player candidates.')
            metadata=copy.deepcopy(metadata)
            metadata['label']='AHR training checkpoint' if metadata.get('optimizer_step',0)>0 else 'Initial AHR weights — no optimizer updates yet'
        elif list(directory.glob('*.pt')):
            raise RuntimeError('Active run has checkpoint payloads without a manifest; no baseline fallback permitted.')
    if model is None:
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(2026100606);model=Go2Network().to(device)
        label='Untrained baseline — awaiting first checkpoint for '+run_id if run_id else 'Untrained baseline — no active training checkpoint yet'
        metadata={'run_id':run_id,'global_step':0,'optimizer_step':0,'is_smoke':False,'training_source':'none',
                  'checkpoint_file':None,'checkpoint_sha256':None,'label':label}
    model.eval()
    for parameter in model.parameters():parameter.requires_grad_(False)
    metadata['model_identity']=go2_model_identity(model);metadata['loaded_at_utc']=go2_now()
    return model,metadata

def go2_play_captures(state):
    root=go2_state_from_board(state.initial_board,to_play=state.initial_to_play,size=state.size,komi=state.komi,
                             positional_history=state.initial_positional_history)
    counts={1:0,-1:0}
    for action in state.actions:
        child=go2_step(root,action);counts[root.to_play]+=root.board.count(-root.to_play)-child.board.count(-root.to_play);root=child
    return counts

class Go2PlaySession:
    def __init__(self,project,model,metadata,human_color=1,search_config=None,game_id=None,persist=True):
        self.project=Path(project);self.model=copy.deepcopy(model).eval()
        for parameter in self.model.parameters():parameter.requires_grad_(False)
        self.metadata=copy.deepcopy(metadata);self.metadata['model_identity']=go2_model_identity(self.model)
        self.human_color=int(human_color)
        if self.human_color not in (-1,0,1):raise ValueError('Choose Black +1, White -1 or watch 0.')
        self.search_config=search_config or Go2SearchConfig(simulations=64,batch_size=8,wall_time_seconds=1.5)
        self.state=go2_initial_state();self.last_search=None;self.events=[];self.sources=go2_play_sources(self.project)
        self.game_id=game_id or 'game-'+uuid.uuid4().hex[:14]
        if Path(self.game_id).name!=self.game_id:raise ValueError('Game ID must be local.')
        self.directory=self.project/'checkpoints/ahr-v2-human-games'/self.game_id
        self.revision=0;self._persist=bool(persist)
        if persist:self.save()

    def save(self):
        if not self._persist:return
        go2_verify_replay(self.state)
        if go2_play_sources(self.project)!=self.sources:raise RuntimeError('Player sources changed during this game.')
        path=self.directory/'game.json'
        with go2_checkpoint_lock(self.directory):
            if path.exists():
                if json.loads(path.read_text()).get('revision')!=self.revision:raise RuntimeError('Another session changed this game; resume before writing.')
            elif self.revision:raise RuntimeError('Saved game disappeared; no silent restart.')
            snapshot=self.directory/'opponent.pt'
            if not snapshot.exists():
                payload={'schema':'ahr-go-v2-player-snapshot-v1','config':dataclasses.asdict(self.model.config),
                         'model':{k:v.detach().cpu().clone() for k,v in self.model.state_dict().items()}}
                buffer=io.BytesIO();torch.save(payload,buffer);go2_atomic_bytes(snapshot,buffer.getvalue())
            record={'schema':GO2_PLAY_SCHEMA,'game_id':self.game_id,'revision':self.revision+1,'sources':self.sources,
                    'model_identity':self.metadata['model_identity'],'snapshot_file':'opponent.pt','snapshot_sha256':go2_file_sha(snapshot),
                    'metadata':self.metadata,'human_color':self.human_color,'search_config':dataclasses.asdict(self.search_config),
                    'state':go2_state_to_dict(self.state),'state_sha256':go2_state_digest(self.state),'events':self.events,
                    'last_search':self.last_search,'updated_at_utc':go2_now()}
            record['payload_sha256']=go2_digest(record);go2_atomic_json(path,record);self.revision=record['revision']
            if not self.metadata.get('is_smoke'):
                go2_atomic_json(self.project/'reports/ahr_v2/latest_human_game.json',
                                {'game_path':str(path.relative_to(self.project)),'updated_at_utc':record['updated_at_utc']})

    @classmethod
    def restore(cls,project,game_path,device='cpu'):
        project=Path(project);path=Path(game_path)
        if not path.is_absolute():path=project/path
        path=path.resolve();base=(project/'checkpoints/ahr-v2-human-games').resolve()
        if not path.is_relative_to(base) or path.name!='game.json':raise ValueError('Resume a saved AHR v2 human game.')
        record=json.loads(path.read_text());checksum=record.pop('payload_sha256',None)
        if checksum!=go2_digest(record) or record.get('schema')!=GO2_PLAY_SCHEMA:raise RuntimeError('Corrupt or unsupported game record.')
        if record['sources']!=go2_play_sources(project):raise RuntimeError('Saved game source identity differs; no silent restart.')
        if record['snapshot_file']!='opponent.pt':raise RuntimeError('Invalid opponent snapshot filename.')
        snapshot=path.parent/'opponent.pt'
        if go2_file_sha(snapshot)!=record['snapshot_sha256']:raise RuntimeError('Frozen opponent snapshot is corrupt.')
        payload=torch.load(snapshot,map_location='cpu',weights_only=True)
        if payload['schema']!='ahr-go-v2-player-snapshot-v1':raise RuntimeError('Wrong opponent snapshot schema.')
        with torch.random.fork_rng(devices=[]):model=Go2Network(Go2NetConfig(**payload['config'])).to(device)
        model.load_state_dict(payload['model']);model.eval()
        if go2_model_identity(model)!=record['model_identity']:raise RuntimeError('Opponent identity differs.')
        state=go2_state_from_dict(record['state']);go2_verify_replay(state)
        if go2_state_digest(state)!=record['state_sha256']:raise RuntimeError('Saved Go history differs.')
        session=cls(project,model,record['metadata'],human_color=record['human_color'],
                    search_config=Go2SearchConfig(**record['search_config']),game_id=record['game_id'],persist=False)
        session.state=state;session.events=record['events'];session.last_search=record['last_search']
        session.revision=record['revision'];session._persist=True
        return session

    def move(self,action,actor='human'):
        if actor=='human' and self.human_color and self.state.to_play!=self.human_color:raise ValueError('It is AHR’s turn; use AI move.')
        child=go2_step(self.state,int(action))
        self.events.append({'move':child.move_number,'color':self.state.to_play,'action':int(action),
                            'coordinate':go2_play_coord(action,self.state.size),'actor':actor})
        self.state=child;self.save();return self.state

    def ai_move(self):
        if self.state.terminal:raise ValueError('Game ended after two passes.')
        config=dataclasses.replace(self.search_config,seed=self.search_config.seed+self.state.move_number)
        result=go2_search(self.model,self.state,config,project=self.project)
        if result['action'] not in go2_legal_actions(self.state):raise AssertionError('AHR selected an illegal action.')
        self.last_search={k:result[k] for k in ('action','status','search_status','completed_simulations','elapsed_seconds','value')}
        self.move(result['action'],actor='ahr');return self.last_search

    def undo(self):
        if not self.state.actions:return
        actions=list(self.state.actions);actions.pop()
        if self.human_color and actions and self.state.initial_to_play*(-1 if len(actions)%2 else 1)!=self.human_color:actions.pop()
        root=go2_state_from_board(self.state.initial_board,to_play=self.state.initial_to_play,size=self.state.size,komi=self.state.komi,
                                  positional_history=self.state.initial_positional_history)
        self.state=go2_replay(root,actions);self.events=self.events[:len(actions)];self.last_search=None;self.save()

    def summary(self):
        score=go2_score(self.state);captures=go2_play_captures(self.state)
        if self.state.terminal:
            winner={1:'Black',-1:'White',0:'Draw'}[go2_outcome(self.state)['winner']]
            status=('Draw' if winner=='Draw' else winner+' wins by '+format(abs(score['score_black']),'.1f'))+' · two consecutive passes'
        else:status=('Black' if self.state.to_play==1 else 'White')+f' to play · move {self.state.move_number} · passes {self.state.consecutive_passes}/2'
        return {'status':status,'captures_black':captures[1],'captures_white':captures[-1],'black_area':score['black_area'],
                'white_area':score['white_area'],'komi':self.state.komi,'terminal':self.state.terminal,'score_black':score['score_black']}


## Clickable board and controls

In [ ]:
def go2_play_static_board(state,last_action=None):
    size=state.size;step=43;margin=34;edge=margin+(size-1)*step;extent=edge+margin
    parts=[f'<svg role="img" aria-label="9 by 9 Go board" width="{extent}" height="{extent}" viewBox="0 0 {extent} {extent}">',
           f'<rect width="{extent}" height="{extent}" rx="10" fill="#ddb47b"/>']
    for index in range(size):
        pos=margin+index*step
        parts += [f'<path d="M {margin} {pos} H {edge} M {pos} {margin} V {edge}" stroke="#563b24" stroke-width="1"/>',
                  f'<text x="{pos}" y="20" font-size="13" text-anchor="middle">{"ABCDEFGHJKLMNOPQRST"[index]}</text>',
                  f'<text x="15" y="{pos+5}" font-size="13" text-anchor="middle">{size-index}</text>']
    for row,col in ((2,2),(2,6),(4,4),(6,2),(6,6)):
        parts.append(f'<circle cx="{margin+col*step}" cy="{margin+row*step}" r="3" fill="#563b24"/>')
    for point,color in enumerate(state.board):
        if color:
            row,col=divmod(point,size);x,y=margin+col*step,margin+row*step
            parts.append(f'<circle cx="{x}" cy="{y}" r="18" fill="{"#1f2328" if color==1 else "#fffdf7"}" stroke="#4f453c"/>')
            if point==last_action:parts.append(f'<circle cx="{x}" cy="{y}" r="4" fill="#d94b3b"/>')
    return ''.join(parts)+'</svg>'

class Go2PlayUI:
    def __init__(self,project,model,metadata,device='cpu',persist=True):
        self.project=Path(project);self.model=model;self.metadata=metadata;self.device=device;self.persist=persist;self.busy=False
        pointer=self.project/'reports/ahr_v2/latest_human_game.json'
        self.resume_path=json.loads(pointer.read_text())['game_path'] if pointer.exists() else None
        self.session=Go2PlaySession(project,model,metadata,persist=persist)
        self.status=widgets.HTML();self.detail=widgets.HTML();self.notice=widgets.HTML()
        self.color=widgets.Dropdown(options=[('Play Black',1),('Play White',-1),('Watch AHR',0)],value=1,layout=widgets.Layout(width='145px'))
        self.which=widgets.Dropdown(options=['latest','best'],value='latest',description='Weights',layout=widgets.Layout(width='190px'))
        self.auto=widgets.Checkbox(value=True,description='AHR replies automatically',indent=False)
        self.visits=widgets.IntSlider(value=64,min=8,max=512,step=8,description='Visits',continuous_update=False,layout=widgets.Layout(width='255px'))
        self.seconds=widgets.FloatSlider(value=1.5,min=.25,max=10,step=.25,description='Seconds',readout_format='.2f',continuous_update=False,layout=widgets.Layout(width='255px'))
        names=('New game','Pass','AI move','Undo turn','Reload checkpoint','Resume saved game')
        self.buttons={name:widgets.Button(description=name,layout=widgets.Layout(width='160px')) for name in names}
        operations=(self.new_game,lambda:self.human_move(81),self.ai_move,lambda:self.session.undo(),self.reload_checkpoint,self.resume_latest)
        for name,operation in zip(names,operations):self.buttons[name].on_click(lambda _,fn=operation:self._perform(fn))
        self.points=[];children=[widgets.HTML('')]+[widgets.HTML('<b>'+c+'</b>') for c in 'ABCDEFGHJ']
        for row in range(9):
            children.append(widgets.HTML('<b>'+str(9-row)+'</b>'))
            for col in range(9):
                action=row*9+col;button=widgets.Button(description='',layout=widgets.Layout(width='42px',height='42px',padding='0px'))
                button.add_class('ahr-go-point');button.tooltip=go2_play_coord(action);button.style.font_size='32px'
                button.on_click(lambda _,a=action:self._perform(lambda:self.human_move(a)))
                self.points.append(button);children.append(button)
        grid=widgets.GridBox(children,layout=widgets.Layout(grid_template_columns='24px '+('42px '*9),grid_gap='0px',width='402px'))
        self.widget=widgets.VBox([widgets.HTML('<h3 style="margin:0">Play AHR · 9×9</h3>'),self.status,grid,
            widgets.HBox([self.color,self.which]),widgets.HBox([self.buttons[k] for k in names[:3]]),
            widgets.HBox([self.buttons[k] for k in names[3:]]),self.auto,widgets.HBox([self.visits,self.seconds]),self.detail,self.notice],
            layout=widgets.Layout(width='580px',max_width='100%'))
        self.render()

    def _perform(self,operation):
        if self.busy:return
        self.busy=True;self.notice.value='Working…';self.render()
        try:operation();self.notice.value=''
        except Exception as error:self.notice.value='<span style="color:#a52c24">'+html.escape(str(error))+'</span>'
        finally:self.busy=False;self.render()

    def _budget(self):
        self.session.search_config=dataclasses.replace(self.session.search_config,simulations=int(self.visits.value),wall_time_seconds=float(self.seconds.value))

    def new_game(self):
        if self.persist and self.session.state.actions:self.resume_path=str((self.session.directory/'game.json').relative_to(self.project))
        self.session=Go2PlaySession(self.project,self.model,self.metadata,human_color=self.color.value,persist=self.persist)
        if self.auto.value and self.session.human_color==-1:self.ai_move()

    def human_move(self,action):
        if self.session.human_color==0:raise ValueError('Watch mode: use AI move, or start a new game as Black/White.')
        self.session.move(action,actor='human')
        if self.auto.value and not self.session.state.terminal:self.ai_move()

    def ai_move(self):
        if self.session.human_color and self.session.state.to_play==self.session.human_color:raise ValueError('It is your turn. Click a point or Pass.')
        self._budget();self.session.ai_move()

    def reload_checkpoint(self):
        self.model,self.metadata=go2_play_load_model(self.project,which=self.which.value,device=self.device);self.new_game()

    def resume_latest(self):
        if not self.resume_path:raise ValueError('No earlier saved human game exists yet.')
        session=Go2PlaySession.restore(self.project,self.resume_path,device=self.device)
        self.session=session;self.model=session.model;self.metadata=session.metadata;self.color.value=session.human_color
        self.visits.value=min(512,max(8,session.search_config.simulations));self.seconds.value=session.search_config.wall_time_seconds or 1.5

    def render(self):
        session=self.session;summary=session.summary();state=session.state;legal=set(go2_legal_actions(state))
        human_turn=bool(session.human_color and state.to_play==session.human_color);last=state.actions[-1] if state.actions else None
        for action,button in enumerate(self.points):
            color=state.board[action];button.description='●' if color else '·' if action in (20,24,40,56,60) else ''
            button.style.text_color='#111827' if color!=-1 else '#fffdf7'
            button.style.button_color='#ddb47b' if action!=last else '#efc875'
            button.disabled=self.busy or not human_turn or action not in legal
            button.tooltip=go2_play_coord(action)+(' · Black' if color==1 else ' · White' if color==-1 else '')
        for button in self.buttons.values():button.disabled=self.busy
        self.buttons['Pass'].disabled=self.busy or not human_turn or state.terminal
        self.buttons['AI move'].disabled=self.busy or human_turn or state.terminal
        self.buttons['Undo turn'].disabled=self.busy or not state.actions
        mode={1:'You: Black',-1:'You: White',0:'Watch mode'}[session.human_color]
        label=html.escape(session.metadata.get('label','AHR checkpoint'));step=session.metadata.get('optimizer_step',0)
        self.status.value='<b>'+html.escape(summary['status'])+'</b><br>'+mode+' · '+label+' · optimizer step '+str(step)
        kind='Final area' if state.terminal else 'Current board area'
        detail=f'{kind}: Black {summary["black_area"]}; White {summary["white_area"]} + {summary["komi"]} komi. Captures: Black {summary["captures_black"]}, White {summary["captures_white"]}.'
        if not state.terminal:detail+=' Capture dead stones before passing.'
        if session.last_search:
            s=session.last_search;detail+=f'<br>Last AHR move: {go2_play_coord(s["action"])} · {s["completed_simulations"]} visits in {s["elapsed_seconds"]:.2f}s · outcome UNKNOWN ({s["status"]}).'
        detail+='<br>Opponent frozen for this game: '+html.escape(session.metadata['model_identity'][:12])+'. Strength is not certified.'
        if self.persist:detail+='<br>Saved game: <code>'+html.escape(str((session.directory/'game.json').relative_to(self.project)))+'</code>'
        self.detail.value=detail


## Bounded control and recovery checks

In [ ]:
def go2_play_validation(project):
    start=time.monotonic();checks={}
    with torch.random.fork_rng(devices=[]):torch.manual_seed(9406);model=Go2Network(Go2NetConfig(width=16,blocks=1))
    model.eval();metadata={'label':'Disposable interface fixture — untrained','optimizer_step':0,'is_smoke':True}
    session=Go2PlaySession(project,model,metadata,search_config=Go2SearchConfig(simulations=8,batch_size=2,wall_time_seconds=None))
    session.move(40);session.ai_move();saved=session.state
    checks['actual_neural_move_legal']=len(saved.actions)==2 and session.last_search['completed_simulations']==8
    restored=Go2PlaySession.restore(project,session.directory/'game.json')
    checks['state_history_and_model_exact_resume']=restored.state==saved and go2_model_identity(restored.model)==go2_model_identity(session.model)
    # A genuinely fresh kernel reads only tagged definitions and the saved game.
    import subprocess,sys
    token=uuid.uuid4().hex[:10];fixture_dir=Path(project)/'reports/ahr_v2/fixtures';fixture_dir.mkdir(parents=True,exist_ok=True)
    fixture=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR Go','language':'python'}})
    setup="from pathlib import Path\nimport nbformat\nPROJECT=Path("+repr(str(project))+")\nfor cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:\n    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())\nfor name in ('01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb','04_selfplay_and_training.ipynb','06_play_and_watch.ipynb'):\n    path=PROJECT/GO2_ROOT/name\n    if path.exists():go2_import_exports(path,globals())\ntorch.set_num_threads(2)\n"
    destination=fixture_dir/('play-resume-'+token+'.json')
    continuation="resumed=Go2PlaySession.restore(PROJECT,"+repr(str(session.directory/'game.json'))+")\ngo2_atomic_json(Path("+repr(str(destination))+"), {'state_sha256':go2_state_digest(resumed.state),'model_identity':go2_model_identity(resumed.model),'revision':resumed.revision,'events':resumed.events})"
    fixture.cells=[nbformat.v4.new_code_cell(setup),nbformat.v4.new_code_cell(continuation)]
    fixture_path=fixture_dir/('play-resume-'+token+'.ipynb');nbformat.write(fixture,fixture_path)
    completed=subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=120',str(fixture_path)],cwd=project,capture_output=True,text=True,timeout=150)
    if completed.returncode:raise RuntimeError(completed.stderr[-5000:])
    fresh=json.loads(destination.read_text())
    checks['fresh_kernel_human_game_resume']=fresh=={'state_sha256':go2_state_digest(saved),'model_identity':go2_model_identity(session.model),'revision':session.revision,'events':session.events}
    before=go2_model_identity(restored.model)
    with torch.no_grad():next(model.parameters()).add_(1)
    checks['opponent_frozen_after_original_weight_change']=go2_model_identity(restored.model)==before
    restored.undo();checks['undo_restores_human_turn']=not restored.state.actions and restored.state.to_play==1
    try:session.move(0)
    except RuntimeError:checks['concurrent_stale_writer_rejected']=True
    else:checks['concurrent_stale_writer_rejected']=False
    terminal=Go2PlaySession(project,restored.model,metadata,human_color=0,persist=False)
    terminal.move(81,actor='ahr');checks['one_pass_not_terminal']=not terminal.state.terminal
    terminal.move(81,actor='ahr');checks['two_passes_white_komi_terminal']=terminal.state.terminal and go2_outcome(terminal.state)['winner']==-1
    terminal.undo();checks['terminal_undo_reopens_game']=not terminal.state.terminal
    controller=Go2PlayUI(project,restored.model,metadata,persist=False);controller.auto.value=False
    controller.points[40].click();checks['click_callback_places_human_stone']=controller.session.state.board[40]==1
    controller.buttons['Undo turn'].click();checks['undo_button_callback']=not controller.session.state.actions
    controller.color.value=0;controller.buttons['New game'].click();controller.visits.value=8;controller.seconds.value=.25
    controller.buttons['AI move'].click();checks['watch_ai_button_callback']=controller.session.state.move_number==1
    checks['coordinates_skip_I']=go2_play_coord(8)=='J9' and go2_play_coord(72)=='A1' and go2_play_coord(81)=='Pass'
    path=restored.directory/'game.json';original=path.read_bytes();record=json.loads(original);record['state']['board'][0]=1
    go2_atomic_json(path,record)
    try:Go2PlaySession.restore(project,path)
    except RuntimeError:checks['corrupt_record_rejected']=True
    else:checks['corrupt_record_rejected']=False
    go2_atomic_bytes(path,original)
    record=json.loads(original);record['sources']['changed']='bad';record.pop('payload_sha256');record['payload_sha256']=go2_digest(record)
    go2_atomic_json(path,record)
    try:Go2PlaySession.restore(project,path)
    except RuntimeError:checks['source_change_rejected']=True
    else:checks['source_change_rejected']=False
    go2_atomic_bytes(path,original)
    result={'schema':'ahr-go-v2-play-validation-v1','status':'PASS' if all(checks.values()) else 'FAIL','checks':checks,
            'sources':go2_play_sources(project),'seconds':time.monotonic()-start,
            'claim':'Interface/replay/model-snapshot correctness; no playing strength claim.'}
    if not all(checks.values()):raise AssertionError(result)
    go2_atomic_json(Path(project)/'reports/ahr_v2/play_validation.json',result);return result


In [ ]:
PLAY_VALIDATION=go2_play_validation(PROJECT)
display(HTML('<table><tr><th>Control/recovery check</th><th>Result</th></tr>'+''.join('<tr><td>'+html.escape(k.replace('_',' '))+'</td><td>PASS</td></tr>' for k in PLAY_VALIDATION['checks'])+'</table>'))


## Play now

In [ ]:
PLAY_MODEL,PLAY_METADATA=go2_play_load_model(PROJECT,which=CHECKPOINT_CHOICE,device=PLAY_DEVICE)
PLAY=Go2PlayUI(PROJECT,PLAY_MODEL,PLAY_METADATA,device=PLAY_DEVICE)
display(HTML("""<style>
.ahr-go-point {margin:0!important;width:42px!important;height:42px!important;padding:0!important;border:0!important;border-radius:0!important;box-shadow:none!important;background-image:linear-gradient(transparent calc(50% - .5px),#775233 calc(50% - .5px),#775233 calc(50% + .5px),transparent calc(50% + .5px)),linear-gradient(90deg,transparent calc(50% - .5px),#775233 calc(50% - .5px),#775233 calc(50% + .5px),transparent calc(50% + .5px));}
.ahr-go-point:disabled {opacity:1!important;}
.ahr-go-point:hover:enabled {outline:2px solid #447c78!important;outline-offset:-2px;}
</style>"""))
display(PLAY.widget)


### Portable board preview

The controls above require a live Jupyter kernel and widget support. This static view preserves the initial board and checkpoint identity in exported documents. The validation fixtures are disposable; this board is your new game.

In [ ]:
summary=PLAY.session.summary()
PREVIEW_HTML='<section style="font-family:system-ui;max-width:650px;padding:20px;border:1px solid #ddd;border-radius:12px"><h2>Play AHR · 9×9</h2><p>'+html.escape(PLAY_METADATA['label'])+' · optimizer step '+str(PLAY_METADATA.get('optimizer_step',0))+'</p>'+go2_play_static_board(PLAY.session.state)+'<p>'+html.escape(summary['status'])+'</p><p>Choose Black/White, click New game, then a board point. Pass · AI move · Undo turn · Reload checkpoint · Resume saved game.</p><p>1.5-second default bounded search. Opponent frozen during your game. Two passes end the game; capture dead stones before passing.</p></section>'
display(HTML(PREVIEW_HTML))
go2_atomic_bytes(PROJECT/'reports/ahr_v2/play_preview.html',('<!doctype html><meta charset="utf-8"><body style="padding:20px;background:#f8f9fa">'+PREVIEW_HTML+'</body>').encode())
print('Ready. No training runs in this notebook. Saved game:',(PLAY.session.directory/'game.json').relative_to(PROJECT))


## Resume and reload

Click **Reload checkpoint** to load the trainer’s current latest/best model and start a new game. `best` is the training manager’s checkpoint cursor; use notebook 05’s independent matches to assess measured strength. CPU is the default so detached GPU training can continue.

For an older game, run `PLAY.session = Go2PlaySession.restore(PROJECT, "checkpoints/ahr-v2-human-games/<game_id>/game.json", device=PLAY_DEVICE)` followed by `PLAY.render()`. Saved human games stay separate from training/development replay and are never imported into training here. Resume does not require the original training checkpoint to remain retained. A time/depth cutoff remains UNKNOWN, with a legal AI move still available. Partial-board area and Worker estimates do not adjudicate dead stones or prove a winner.